# Variant-to-function evidence agent: Colab runner

Thin front end over the `variant_evidence` package. All logic lives in the repo; this notebook only
fetches data and calls it. **Three pieces are unverified until you run this** (they were written without
network access): the AG2 download format, the Open Targets query, and the held-out gold loci.
Where a cell can fail on that, the error message tells you what to paste back.

Runtime: CPU is enough.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/variant-evidence-data'
import os; os.makedirs(DATA_DIR, exist_ok=True)

REPO = 'https://github.com/Raj4884/Variant-to-function-evidence-agent.git'
if os.path.exists('/content/repo'):
    !git -C /content/repo pull -q
else:
    !git clone -q $REPO /content/repo
%cd /content/repo
!pip install -q -e ".[dev,llm]" duckdb pandas
!python -m pytest -q

## 2. Locate the AG2 files
Files come from the AlphaGenome 2 predictions download (non-commercial use only). They have to be in your Drive.
This cell searches your Drive for them, wherever you put them.

In [ ]:
import glob
def find(name):
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    assert hits, f'{name} not found in Drive; is it uploaded and the mount cell run?'
    return hits[0]
EXPR = find('variant_scores_sc_gene_expression.parquet')
VARIANTS = find('variants.parquet')
print(EXPR); print(VARIANTS)

## 3. Inspect (expected columns are in the release README; confirm they match)

In [ ]:
import duckdb
con = duckdb.connect()
print(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{EXPR}')").fetchdf()[['column_name','column_type']])
print(con.execute(f"SELECT * FROM read_parquet('{EXPR}') LIMIT 3").fetchdf().T)
print(con.execute(f"SELECT DISTINCT chromosome FROM read_parquet('{VARIANTS}') LIMIT 5").fetchdf().T)
print(con.execute(f"SELECT channel_name, count(*) n FROM read_parquet('{EXPR}') GROUP BY 1 ORDER BY n DESC").fetchdf())

## 4. rsID -> coordinates -> filter -> normalise
The AG2 files have no rsIDs, so rsIDs are resolved to hg38 coordinates through the Ensembl REST API (unverified call; if it errors, paste the message back).
Check the printed LFC summary: if values look like log2 fold changes, set `LFC_SCALE = math.log(2)` and rerun.

In [ ]:
import math
from variant_evidence.resolve import resolve_rsids
from variant_evidence.convert import normalise_ag2_expression, write_normalised_csv

RSIDS = ['rs2836882','rs12913832','rs6497271','rs1175550','rs6673829','rs2038740',
         'rs13066062','rs35705950','rs3820282','rs1421085','rs12740374']
LFC_SCALE = 1.0

coords = resolve_rsids(RSIDS)
coord_to_rsid = {c: r for r, cs in coords.items() for c in cs}
print({r: len(cs) for r, cs in coords.items()})

positions = sorted({c[1] for c in coord_to_rsid})
pos_list = ','.join(map(str, positions))
cur = con.execute(f"SELECT * FROM read_parquet('{EXPR}') WHERE position IN ({pos_list})")
cols = [d[0] for d in cur.description]
raw = (dict(zip(cols, r)) for r in cur.fetchall())
OUT = '/content/drive/MyDrive/ag2_normalised.csv'
n = write_normalised_csv(normalise_ag2_expression(raw, coord_to_rsid, lfc_scale=LFC_SCALE), OUT)
print(n, 'rows written to', OUT)

In [ ]:
import pandas as pd
df = pd.read_csv(OUT)
print(df.groupby('variant').size())
print(df.lfc.describe())   # the paper's call threshold is |LFC| > 0.1

## 5. Open Targets (live)
Step 1 lists the real field names. If the query in step 2 errors, fix `CREDIBLE_SET_QUERY` and
`rows_from_response` in `src/variant_evidence/providers/open_targets.py` (or paste the output to Claude).

In [ ]:
from variant_evidence.providers import open_targets as ot
for t in ['Variant', 'CredibleSet']:
    try: print(t, ot.introspect(t))
    except Exception as e: print(t, 'ERROR', e)

In [ ]:
print(ot.resolve_variant_id('rs12740374'))
from variant_evidence.models import VariantQuery
try:
    print(ot.http_transport()(VariantQuery(rsid='rs12740374'))[:3])
except Exception as e:
    print('ERROR:', e)

## 6. Run the agent

In [ ]:
from variant_evidence.agent import run_agent, llm_synthesizer
from variant_evidence.models import VariantQuery
from variant_evidence.providers.ag2_precomputed import AG2PrecomputedProvider

USE_OT = False      # set True once step 5 works
USE_LLM = False     # set True to use Claude for wording (key in Colab Secrets as ANTHROPIC_API_KEY)
providers = [AG2PrecomputedProvider(f'/content/drive/MyDrive/ag2_normalised.csv')]
if USE_OT: providers.append(ot.OpenTargetsProvider(ot.http_transport()))
synth = None
if USE_LLM:
    from google.colab import userdata; import anthropic
    synth = llm_synthesizer(anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY')), 'claude-sonnet-5-5')

report = run_agent(VariantQuery(rsid='rs12913832'), providers, synth)
print(report.model_dump_json(indent=2)[:4000])

## 7. Evaluation on the gold loci

In [ ]:
!python eval/run_eval.py --ag2 /content/drive/MyDrive/ag2_normalised.csv

## 8. What to send back
Paste into the chat: (a) the file name and the DESCRIBE + first rows from step 3, (b) any step 5 errors or the
introspection output, (c) the eval summary lines from step 7. The two `heldout_candidate` loci came from memory
and must be checked against the literature before any number is quoted.